In [ ]:
import pandas as pd
import re
from google import genai
from dotenv import load_dotenv
import json
import re
import ast

load_dotenv('../../backend/.env')
client = genai.Client()
def get_payments_df(transactions_df:pd.DataFrame) -> pd.DataFrame:
    """separates actual payments from just transactions"""
    def find_transaction_object(text):
        pattern = r"ობიექტი:\s*([^,]+)"
        res = re.search(pattern, text)
        if res:
            return res.group(1)
    
    payments_df = transactions_df.loc[transactions_df["transaction_type"] == "გადახდა"] # separate actal payments from transactions
    payments_df["transaction_object"] = payments_df["დანიშნულება"].apply(lambda x:find_transaction_object(x) ) # separate where transaction was actually performed
    payments_df["transaction_object"] = payments_df["transaction_object"].fillna(value="gadakhda") # fill wherever object was unavailable
    for currency in ["GEL","USD","EUR","GBP"]:
        payments_df[currency] = payments_df[currency].apply(lambda x: x * -1)
    payments_df.drop(columns=["transaction_type"],inplace=True)
    payments_df.fillna(value=0, inplace=True) # fill other NaN values
    payments_df["transaction_object"] = payments_df["transaction_object"].apply(lambda x: x.lower()) # convert to lowercase
    

    return payments_df


In [ ]:

transactions_df = pd.read_excel("../../data/raw/amonaweri.xlsx", sheet_name="ტრანზაქციები")
transactions_df.drop(columns=["Unnamed: 2"],inplace=True) # drop unneccessary col
transactions_df["transaction_type"] = transactions_df["დანიშნულება"].apply(lambda x: x.split()[0]) # separate by transaction type

df = get_payments_df(transactions_df)
merchants_df = pd.read_csv('../../data/processed/merchants_df.csv')

c:\Users\Saba\.conda\envs\financeAI\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [160]:
df

,თარიღი,დანიშნულება,GEL,USD,EUR,GBP,transaction_object
2,07/03/2025,"გადახდა - თანხა: GEL2.94; ობიექტი: LIBRE, Tbil...",2.94,0.0,0.0,0.0,libre
6,09/03/2025,"გადახდა - თანხა GEL3.00; გადახდა, 09/03/2025 ,...",3.00,0.0,0.0,0.0,gadakhda
12,09/03/2025,"გადახდა - თანხა: GEL5.34; ობიექტი: Nikora, Tbi...",5.34,0.0,0.0,0.0,nikora
13,09/03/2025,"გადახდა - თანხა: GEL68.06; ობიექტი: libre, Tbi...",68.06,0.0,0.0,0.0,libre
14,10/03/2025,გადახდა - თანხა: GEL6.30; ობიექტი: shps gama 2...,6.30,0.0,0.0,0.0,shps gama 2023
...,...,...,...,...,...,...,...
500,04/03/2026,"გადახდა - თანხა: GEL38.97; ობიექტი: XS Toys, T...",38.97,0.0,0.0,0.0,xs toys
501,05/03/2026,"გადახდა - თანხა: GEL3.20; ობიექტი: MAGNITI, Tb...",3.20,0.0,0.0,0.0,magniti
503,05/03/2026,"გადახდა - თანხა GEL7.00; გადახდა, 05/03/2026 ,...",7.00,0.0,0.0,0.0,gadakhda
505,05/03/2026,"გადახდა - თანხა: GEL8.20; ობიექტი: ORI NABIJI,...",8.20,0.0,0.0,0.0,ori nabiji


In [161]:
merchants_df

,merchant,category,confidence
0,libre,Groceries,high
1,gadakhda,Utilities,medium
2,nikora,Groceries,high
3,shps gama 2023,Healthcare,high
4,spar,Groceries,high
5,magniti,Groceries,high
6,ori nabiji,Groceries,high
7,shemogevle,Dining,high
8,ie goderdzi apkhazashvili,Groceries,low
9,tea house veris bagi,Dining,high


In [162]:
keys = merchants_df["merchant"]
values = merchants_df["category"]

merchants_dict = dict(zip(keys,values))
df["category"] = df["transaction_object"].map(merchants_dict)
df["თარიღი"] = pd.to_datetime(df["თარიღი"], dayfirst=True) 
df

,თარიღი,დანიშნულება,GEL,USD,EUR,GBP,transaction_object,category
2,2025-03-07,"გადახდა - თანხა: GEL2.94; ობიექტი: LIBRE, Tbil...",2.94,0.0,0.0,0.0,libre,Groceries
6,2025-03-09,"გადახდა - თანხა GEL3.00; გადახდა, 09/03/2025 ,...",3.00,0.0,0.0,0.0,gadakhda,Utilities
12,2025-03-09,"გადახდა - თანხა: GEL5.34; ობიექტი: Nikora, Tbi...",5.34,0.0,0.0,0.0,nikora,Groceries
13,2025-03-09,"გადახდა - თანხა: GEL68.06; ობიექტი: libre, Tbi...",68.06,0.0,0.0,0.0,libre,Groceries
14,2025-03-10,გადახდა - თანხა: GEL6.30; ობიექტი: shps gama 2...,6.30,0.0,0.0,0.0,shps gama 2023,Healthcare
...,...,...,...,...,...,...,...,...
500,2026-03-04,"გადახდა - თანხა: GEL38.97; ობიექტი: XS Toys, T...",38.97,0.0,0.0,0.0,xs toys,Shopping
501,2026-03-05,"გადახდა - თანხა: GEL3.20; ობიექტი: MAGNITI, Tb...",3.20,0.0,0.0,0.0,magniti,Groceries
503,2026-03-05,"გადახდა - თანხა GEL7.00; გადახდა, 05/03/2026 ,...",7.00,0.0,0.0,0.0,gadakhda,Utilities
505,2026-03-05,"გადახდა - თანხა: GEL8.20; ობიექტი: ORI NABIJI,...",8.20,0.0,0.0,0.0,ori nabiji,Groceries


In [ ]:
def get_spending_by_month(df:pd.DataFrame):
    df["month"] = df["თარიღი"].dt.month
    df["year"] = df["თარიღი"].dt.year
    
    df = df.groupby(["month","year"])[["GEL","USD","EUR","GBP"]].sum()
    return df
def get_spending_by_category(df:pd.DataFrame):
    df = df.groupby(["category"])[["GEL","USD","EUR","GBP"]].sum().sort_values(by="GEL", ascending=False)
    return df

def get_total_spending(df:pd.DataFrame):
    total = df[["GEL","USD","EUR","GBP"]].sum()
    return total

def get_transaction_means(df:pd.DataFrame):
    means = df[["GEL","USD","EUR","GBP"]].mean()
    return means
def get_transaction_medians(df:pd.DataFrame):
    #TODO: make it ignore zeros
    medians = df[["GEL","USD","EUR","GBP"]].median()
    return medians

def get_biggest_spending(df:pd.DataFrame,currency:str="GEL"):
    return df.sort_values(by=currency, ascending=False).head(1)
def get_transaction_count(df:pd.DataFrame):
    count = int(df['თარიღი'].count())
    return count

def get_spending_by_merchant(df:pd.DataFrame):
    df = df.groupby(["transaction_object"])[["GEL","USD","EUR","GBP"]].sum().sort_values(by="GEL", ascending=False)
    return df


def get_transactions_per_day(df:pd.DataFrame):
    return df.groupby(["თარიღი"])[["GEL"]].count().sort_values(by="GEL", ascending=False)

def get_most_active_day(df:pd.DataFrame):
    return get_transactions_per_day(df).head(1)

def get_anomaly_transactions(df:pd.DataFrame, currency:str="GEL"):
    std = df[currency].std()
    mean = df[currency].mean()
    df_anomaly = df[df[currency] > mean + 2 * std]
    return df_anomaly
    
def get_month_comparison_analysis(latest_month_df, previous_month_biggest_spending_category, currency="GEL"):
    if len(previous_month_biggest_spending_category) != 0:
        percent_difference = ((latest_month_df[currency] - previous_month_biggest_spending_category[currency]) / previous_month_biggest_spending_category[currency]) * 100 if previous_month_biggest_spending_category[currency] != 0 else float('inf')
        
        if percent_difference > 0:
            return f"This month your spending on {latest_month_df["category"]} was more than previous month by {percent_difference}%"
        else:
            return f"This month your spending on {latest_month_df["category"]} was less than previous month by {percent_difference}%"
    else:
        return "No previous month data available"

def get_month_category_comparison(df:pd.DataFrame, currency:str="GEL"):
    df["month"] = df["თარიღი"].dt.month
    df["year"] = df["თარიღი"].dt.year
    df = df.groupby(["year","month","category"])[["GEL","USD","EUR","GBP"]].sum()

    df_reset = df.reset_index()

    latest = df_reset.sort_values(by=["year",'month']).iloc[-1]
    latest_month, latest_year = latest[["month","year"]]

    previous_month = latest_month - 1 if latest_month != 1 else 12

    latest_month_df = df_reset[(df_reset["year"] == latest_year) & (df_reset['month'] == latest_month)]
    previous_month_df = df_reset[(df_reset["year"] == latest_year) & (df_reset['month'] == previous_month)]

    latest_month_df = latest_month_df.sort_values(by=[currency], ascending=False).head(1)
    previous_month_df = previous_month_df.sort_values(by=[currency], ascending=False)
    biggest_spending_category = latest_month_df["category"].values[0]
    previous_month_biggest_spending_category = previous_month_df[previous_month_df["category"] == biggest_spending_category].values
    return get_month_comparison_analysis(latest_month_df, previous_month_biggest_spending_category, currency)


    

'No previous month data available'

Percent difference: 73.53%


,year,month,category,GEL,USD,EUR,GBP
42,2026,3,Shopping,118.97,0.0,0.0,0.0


,year,month,category,GEL,USD,EUR,GBP
